# 🪔 Diwali Video Face Swap — FaceFusion on Google Colab (consensual use)

**Input:** `target_video.mp4` (the Diwali video) + `source_face.jpg` (a clear photo of the person whose face is swapped in)
**Output:** `/content/swapped_video.mp4`

**Pipeline:** `source_face.jpg + target_video.mp4 → face detection → face tracking → face swap → mask blending → output video`

### Why FaceFusion?
[FaceFusion](https://github.com/facefusion/facefusion) (pinned to release **3.9.2**, Oct 2026) is the most actively maintained open-source
*frame-based* face swapper. It uses **pre-trained ONNX models** (no training), runs on the GPU through `onnxruntime-gpu`,
and has a headless CLI that suits Colab. Its official installer expects conda, so this notebook installs it into an isolated
Python 3.12 `uv` virtual environment with the **CUDA 12 build of onnxruntime-gpu** (CUDA/cuDNN come from pip wheels).
That works on any Colab NVIDIA driver and doesn't touch Colab's own Python packages.

### What it does and doesn't do
* Only the **inner face region** of the chosen person is replaced. Body, clothes, hair, background and camera motion come straight from the original frames.
* A frame with **no detectable face** (back of head, face out of shot) is **left unchanged**. Nothing is generated. Only faces that match the chosen person's identity are swapped.
* Duration, FPS and resolution are preserved, and the **original audio track** is muxed back in at the end.
* It is **not** a generative video model. Each frame is processed independently by the face-swap network.

### ⚠️ Consent
Use this only with the **explicit consent of everyone whose face appears** (the person in the photo and the person in the video).
Do not use it to deceive, harass, impersonate or create sexual content. FaceFusion's built-in content filter stays enabled.

### How to run
1. `Runtime → Change runtime type → GPU` (T4 / L4 / A100).
2. `Runtime → Run all`, or run the cells top to bottom. Two cells ask you to **upload a file** (marked 📤).
3. Check the **5-second preview** (section 6). If it looks right, the full run continues automatically.

**After a session restart or disconnect:** run all cells again. Uploads, model weights and finished video chunks are kept
(on Google Drive if enabled), so the work resumes where it stopped instead of starting over.

## 1 · GPU / environment setup

In [ ]:
#@title 1.1 · Check the GPU (nvidia-smi) { display-mode: "form" }
# Shows the GPU name, VRAM, driver and the max CUDA version the driver supports.
# Stops early with a clear message if no GPU runtime is selected.
import re, shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('❌ No NVIDIA GPU found. Use Runtime → Change runtime type → GPU, then run again.')

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(smi)
query = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'],
                       capture_output=True, text=True).stdout.strip().splitlines()[0]
GPU_NAME, GPU_VRAM_MB, GPU_DRIVER = [x.strip() for x in query.split(',')]
GPU_VRAM_GB = round(int(float(GPU_VRAM_MB)) / 1024, 1)
cuda_match = re.search(r'CUDA Version:\s*([\d.]+)', smi)
GPU_CUDA = cuda_match.group(1) if cuda_match else 'unknown'

print(f'✅ GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB} GB | driver {GPU_DRIVER} | driver CUDA {GPU_CUDA}')
if GPU_CUDA != 'unknown' and float(GPU_CUDA.split('.')[0]) < 12:
    raise SystemExit('❌ This driver is older than CUDA 12. Pick a different GPU runtime.')

In [ ]:
#@title 1.2 · Configuration + helpers — re-run after every restart { display-mode: "form" }
# All settings live here. The defaults favour quality and robustness to profile / occluded faces.

# ---- Storage -------------------------------------------------------------------------------
USE_GOOGLE_DRIVE = True          #@param {type:"boolean"}
DRIVE_FOLDER = "diwali_faceswap" #@param {type:"string"}
# Drive keeps uploads, model weights (~1.5 GB) and finished chunks across runtime resets.

# ---- Face swap model -----------------------------------------------------------------------
FACE_SWAPPER_MODEL = "hyperswap_1a_256"  #@param ["hyperswap_1a_256", "hyperswap_1b_256", "hyperswap_1c_256", "inswapper_128_fp16", "simswap_unofficial_512", "ghost_2_256"]
PIXEL_BOOST = "512x512"                  #@param ["256x256", "512x512", "768x768", "1024x1024"]
# hyperswap_1a_256 is FaceFusion's current default. Pixel boost runs the model on tiles for sharper, high-res faces.
FACE_ENHANCER = False                    #@param {type:"boolean"}
FACE_ENHANCER_MODEL = "gfpgan_1.4"       #@param ["gfpgan_1.4", "gpen_bfr_512", "codeformer", "restoreformer_plus_plus"]
FACE_ENHANCER_BLEND = 50                 #@param {type:"slider", min:0, max:100, step:5}
# Enhancer = optional extra sharpening of the swapped face. Can soften identity; compare in the preview.
EXPRESSION_RESTORER = False              #@param {type:"boolean"}
EXPRESSION_RESTORER_FACTOR = 80          #@param {type:"slider", min:0, max:100, step:5}
# Expression restorer (LivePortrait) puts the ORIGINAL expression / mouth / eye movement back onto the swapped
# face. Often makes head turns and smiles look more natural. Slower (~1.5-2x).
# ⚠️ On strong head tilts (looking up at the sky, filmed from below) it can warp the face badly
#    ("melted" look). Check those moments in the preview; leave it off if they look wrong.

# ---- Detection / tracking (profiles, occlusion) -------------------------------------------
FACE_DETECTOR_MODEL = "many"             #@param ["many", "yolo_face", "retinaface", "scrfd", "yunet"]
FACE_DETECTOR_SCORE = 0.45               #@param {type:"slider", min:0.2, max:0.9, step:0.05}
FACE_LANDMARKER_MODEL = "many"           #@param ["many", "2dfan4", "peppa_wutz", "hrffa"]
FACE_TRACKER_SCORE = 0.3                 #@param {type:"slider", min:0.0, max:0.5, step:0.05}
# "many" = combine several detectors/landmarkers (best for side profiles, slower).
# Lower detector score finds more profile faces but risks false detections.
# Tracker > 0 links faces across neighbouring frames by box overlap, which reduces flicker. It only bridges
# single-frame detection misses BETWEEN real detections, so it never invents a face on a back-of-head shot. 0 = off.

# ---- Which person in the video gets the new face -----------------------------------------
FACE_SELECTOR_MODE = "reference"         #@param ["reference", "one"]
# "reference": swap only faces that match the chosen person (use when several people appear in the video).
# "one": only ONE person is ever in the video → swap the most confident face in every frame, with no identity
#        check. Much better coverage on head turns, where her own face looks too different to match.
REFERENCE_TIME_SECONDS = -1.0            #@param {type:"number"}
REFERENCE_FACE_POSITION = 0              #@param {type:"integer"}
REFERENCE_FACE_DISTANCE = 0.4            #@param {type:"slider", min:0.1, max:0.8, step:0.05}
FACE_SELECTOR_GENDER = "none"            #@param ["none", "female", "male"]
# REFERENCE_TIME_SECONDS = -1 → auto: picks the frame with the largest clear face.
# REFERENCE_FACE_POSITION = number shown on the green box in the face map (0 = largest face).
# REFERENCE_FACE_DISTANCE: how different a face may look (e.g. turned to profile) and still count as the same person.
# Higher = swaps more angles; too high may also swap other people. Lower = stricter.

# ---- Blending ------------------------------------------------------------------------------
OCCLUSION_MASK = True                    #@param {type:"boolean"}
REGION_MASK = False                      #@param {type:"boolean"}
FACE_MASK_BLUR = 0.3                     #@param {type:"slider", min:0.0, max:1.0, step:0.05}
FACE_MASK_PADDING = "0 0 0 0"            #@param {type:"string"}
# Occlusion mask (XSeg) keeps hands, diyas, sparklers, phones etc. that are IN FRONT of the face.
# Region mask (face parser) restricts the swap to skin/eyes/nose/mouth, so hair/fringe over the face stays original.
# Padding = top right bottom left in % (shrinks the box mask).

# ---- Preview / run -------------------------------------------------------------------------
PREVIEW_SECONDS = 5                      #@param {type:"integer"}
PREVIEW_START_SECONDS = -1.0             #@param {type:"number"}
CONTINUE_AFTER_PREVIEW = True            #@param {type:"boolean"}
CHUNK_SECONDS = 20                       #@param {type:"integer"}
OUTPUT_VIDEO_QUALITY = 92                #@param {type:"slider", min:50, max:100, step:1}
EXECUTION_PROVIDER = "cuda"              #@param ["cuda", "cpu"]
# PREVIEW_START_SECONDS = -1 → preview window is centred on the reference frame.
# CONTINUE_AFTER_PREVIEW = False makes "Run all" stop after the preview so you can tweak settings first.
# Full video is processed in chunks; finished chunks are saved, so a disconnect resumes instead of restarting.

FACEFUSION_VERSION = "3.9.2"
BASE_DIR = "/content"
FINAL_OUTPUT = f"{BASE_DIR}/swapped_video.mp4"

# ============================================================================================
import glob, hashlib, json, os, pathlib, re, shutil, subprocess, sys, time
from fractions import Fraction
from IPython.display import Image, display

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.isdir('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        PERSIST_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    except Exception as error:
        print(f'⚠️ Google Drive not mounted ({error}); falling back to local storage (lost on runtime reset).')
        PERSIST_DIR = f'{BASE_DIR}/{DRIVE_FOLDER}'
else:
    PERSIST_DIR = f'{BASE_DIR}/{DRIVE_FOLDER}'

FF_DIR      = f'{BASE_DIR}/facefusion'          # FaceFusion code (local disk, fast)
VENV_DIR    = f'{BASE_DIR}/ffenv'               # isolated Python 3.12 env
FF_PY       = f'{VENV_DIR}/bin/python'
FF_TOOLS    = '/content/ff_tools.py'          # written by cell 2.2
BIN_DIR     = f'{BASE_DIR}/bin'                 # newer static ffmpeg if Colab's is too old
LOCAL_WORK  = f'{BASE_DIR}/faceswap_work'       # scratch (fast local disk)
INPUT_DIR   = f'{PERSIST_DIR}/inputs'
MODELS_DIR  = f'{PERSIST_DIR}/models'
CHUNKS_DIR  = f'{PERSIST_DIR}/chunks'
OUTPUT_DIR  = f'{PERSIST_DIR}/outputs'
for d in (LOCAL_WORK, INPUT_DIR, MODELS_DIR, CHUNKS_DIR, OUTPUT_DIR, BIN_DIR):
    os.makedirs(d, exist_ok=True)
TARGET_META_JSON = f'{PERSIST_DIR}/target_meta.json'
REFERENCE_JSON = f'{PERSIST_DIR}/reference.json'

if os.path.isfile(f'{BIN_DIR}/ffmpeg') and BIN_DIR not in os.environ['PATH'].split(':'):
    os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']


def ff_env():
    """Environment for FaceFusion subprocesses: venv first on PATH + CUDA/cuDNN libs from pip wheels."""
    env = dict(os.environ)
    libs = sorted(glob.glob(f'{VENV_DIR}/lib/python3*/site-packages/nvidia/*/lib'))
    env['LD_LIBRARY_PATH'] = ':'.join(libs + [env.get('LD_LIBRARY_PATH', '')]).strip(':')
    env['PATH'] = f'{VENV_DIR}/bin:' + env['PATH']
    env['PYTHONUNBUFFERED'] = '1'
    env.pop('MPLBACKEND', None)  # Colab sets a backend the venv doesn't have
    return env


def sh(cmd, env=None, check=True, quiet=False, cwd=None):
    """Run a command, streaming its output live. Returns (returncode, captured_output)."""
    if isinstance(cmd, str):
        cmd = ['bash', '-lc', cmd]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1, cwd=cwd)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet:
            print(line, end='')
    proc.wait()
    if check and proc.returncode != 0:
        tail = ''.join(lines[-30:])
        raise RuntimeError(f'Command failed ({proc.returncode}): {" ".join(map(str, cmd))[:300]}\n{tail}')
    return proc.returncode, ''.join(lines)


def ff_tool(*args, quiet=True):
    """Call ff_tools.py inside the FaceFusion venv and return its JSON result."""
    _, out = sh([FF_PY, FF_TOOLS, FF_DIR, *map(str, args)], env=ff_env(), quiet=quiet)
    for line in out.splitlines()[::-1]:
        if line.startswith('@@JSON@@'):
            return json.loads(line[len('@@JSON@@'):])
    raise RuntimeError('ff_tools gave no result:\n' + out[-3000:])


def ffprobe(path):
    _, out = sh(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_format', '-show_streams', path], quiet=True)
    return json.loads(out)


def count_frames(path):
    _, out = sh(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-count_packets',
                 '-show_entries', 'stream=nb_read_packets', '-of', 'csv=p=0', path], quiet=True)
    return int(out.strip().splitlines()[0])


def file_signature(path):
    st = os.stat(path)
    return f'{os.path.basename(path)}:{st.st_size}:{int(st.st_mtime)}'


def find_input(stem):
    hits = sorted(p for p in glob.glob(f'{INPUT_DIR}/{stem}.*') if not p.endswith('.part'))
    return hits[0] if hits else None


def gpu_tuning():
    """Threads / memory strategy scaled to the GPU's VRAM."""
    if EXECUTION_PROVIDER != 'cuda':
        return 4, 'strict'
    try:
        vram = GPU_VRAM_GB
    except NameError:
        vram = 15
    if vram >= 38: return 16, 'tolerant'
    if vram >= 20: return 12, 'tolerant'
    return 8, 'moderate'


def facefusion_args(reference_frame=0):
    """All FaceFusion CLI options derived from the settings above (except paths / trimming)."""
    threads, memory_strategy = gpu_tuning()
    processors = ['face_swapper'] + (['expression_restorer'] if EXPRESSION_RESTORER else []) + (['face_enhancer'] if FACE_ENHANCER else [])
    mask_types = ['box'] + (['occlusion'] if OCCLUSION_MASK else []) + (['region'] if REGION_MASK else [])
    args = [
        '--processors', *processors,
        '--face-swapper-model', FACE_SWAPPER_MODEL,
        '--face-swapper-pixel-boost', PIXEL_BOOST,
        '--face-detector-model', FACE_DETECTOR_MODEL,
        '--face-detector-score', str(FACE_DETECTOR_SCORE),
        '--face-landmarker-model', FACE_LANDMARKER_MODEL,
        '--face-tracker-score', str(FACE_TRACKER_SCORE),
        '--face-selector-mode', FACE_SELECTOR_MODE,
        '--face-selector-order', 'best-worst' if FACE_SELECTOR_MODE == 'one' else 'large-small',
        '--reference-frame-number', str(int(reference_frame)),
        '--reference-face-position', str(int(REFERENCE_FACE_POSITION)),
        '--reference-face-distance', str(REFERENCE_FACE_DISTANCE),
        '--face-mask-types', *mask_types,
        '--face-occluder-model', 'xseg_2',
        '--face-parser-model', 'bisenet_resnet_34',
        '--face-mask-blur', str(FACE_MASK_BLUR),
        '--face-mask-padding', *FACE_MASK_PADDING.split(),
        '--output-video-encoder', 'libx264',
        '--output-video-preset', 'fast',
        '--output-video-quality', str(int(OUTPUT_VIDEO_QUALITY)),
        '--execution-providers', EXECUTION_PROVIDER,
        '--execution-thread-count', str(threads),
        '--video-memory-strategy', memory_strategy,
        '--download-providers', 'github', 'huggingface',
        '--temp-path', f'{LOCAL_WORK}/ff_temp',
        '--log-level', 'info',
    ]
    if FACE_SELECTOR_GENDER != 'none':
        args += ['--face-selector-gender', FACE_SELECTOR_GENDER]
    if EXPRESSION_RESTORER:
        args += ['--expression-restorer-model', 'live_portrait', '--expression-restorer-factor', str(int(EXPRESSION_RESTORER_FACTOR))]
    if FACE_ENHANCER:
        args += ['--face-enhancer-model', FACE_ENHANCER_MODEL, '--face-enhancer-blend', str(int(FACE_ENHANCER_BLEND))]
    return args


RUNTIME_ONLY_ARGS = {'--execution-providers', '--execution-thread-count', '--video-memory-strategy',
                     '--download-providers', '--temp-path', '--log-level'}


def pixel_affecting_args(args):
    """facefusion_args() minus options that change speed but not the output pixels."""
    kept, skip = [], False
    for item in args:
        if item.startswith('--'):
            skip = item in RUNTIME_ONLY_ARGS
        if not skip:
            kept.append(item)
    return kept


def run_facefusion(sources, target, output, trim_start, trim_end, reference_frame):
    sources = [sources] if isinstance(sources, str) else list(sources)
    cmd = [FF_PY, 'facefusion.py', 'headless-run', '-s', *sources, '-t', target, '-o', output,
           '--trim-frame-start', str(trim_start), '--trim-frame-end', str(trim_end)] + facefusion_args(reference_frame)
    started = time.time()
    proc = subprocess.Popen(cmd, cwd=FF_DIR, env=ff_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log = []
    for line in proc.stdout:
        log.append(line)
        # tqdm redraws with \r; only print meaningful lines to keep the cell output short
        if '[FACEFUSION' in line or 'Error' in line or 'error' in line:
            print('   ' + line.strip()[-200:])
    proc.wait()
    if proc.returncode != 0 or not os.path.isfile(output):
        raise RuntimeError(f'FaceFusion failed (exit {proc.returncode}). Last log lines:\n' + ''.join(log[-40:]))
    return time.time() - started


def show_video(path, width=900):
    """Embed a (small) mp4 inline in the notebook."""
    from base64 import b64encode
    from IPython.display import HTML, display
    data = b64encode(open(path, 'rb').read()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))


print('✅ Settings loaded')
print('   Persistent folder :', PERSIST_DIR)
print('   Final output      :', FINAL_OUTPUT)

## 2 · Dependency installation

In [ ]:
#@title 2.1 · Install FaceFusion + GPU runtime (≈3–5 min the first time, seconds after a session restart) { display-mode: "form" }
# 1) ffmpeg ≥ 5.1 (FaceFusion uses -fps_mode). Downloads a static build only if Colab's is older.
# 2) FaceFusion source, pinned to a release tag for reproducibility.
# 3) Isolated Python 3.12 env (uv) with FaceFusion requirements + onnxruntime-gpu (CUDA 12 build)
#    and its CUDA/cuDNN wheels. gradio (the web UI) is skipped because this notebook uses the headless CLI.
# 4) The FaceFusion model folder is linked to persistent storage, so weights survive runtime resets.

# ---- 1) ffmpeg ----
def ffmpeg_version():
    try:
        out = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout
        m = re.search(r'ffmpeg version n?(\d+)\.(\d+)', out)
        return (int(m.group(1)), int(m.group(2))) if m else (99, 0)   # git builds: assume new
    except FileNotFoundError:
        return (0, 0)

if ffmpeg_version() < (5, 1):
    print('Installing a recent static ffmpeg build...')
    url = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz'
    sh(f'cd {LOCAL_WORK} && curl -sSLf -o ffmpeg.tar.xz {url} && rm -rf ffmpeg_static && mkdir ffmpeg_static '
       f'&& tar -xf ffmpeg.tar.xz -C ffmpeg_static --strip-components=1 '
       f'&& cp ffmpeg_static/bin/ffmpeg ffmpeg_static/bin/ffprobe {BIN_DIR}/ && rm -rf ffmpeg.tar.xz ffmpeg_static')
    os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('ffmpeg:', subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout.splitlines()[0])

# ---- 2) FaceFusion source ----
def facefusion_tag():
    r = subprocess.run(['git', '-C', FF_DIR, 'describe', '--tags', '--exact-match'], capture_output=True, text=True)
    return r.stdout.strip()

if not os.path.isdir(FF_DIR) or facefusion_tag() != FACEFUSION_VERSION:
    shutil.rmtree(FF_DIR, ignore_errors=True)
    sh(['git', 'clone', '--depth', '1', '--branch', FACEFUSION_VERSION, 'https://github.com/facefusion/facefusion.git', FF_DIR])
print('FaceFusion', facefusion_tag())

# models → persistent storage
assets = f'{FF_DIR}/.assets'
os.makedirs(assets, exist_ok=True)
models_link = f'{assets}/models'
if os.path.islink(models_link) and os.readlink(models_link) != MODELS_DIR:
    os.unlink(models_link)
if os.path.isdir(models_link) and not os.path.islink(models_link):
    for f in glob.glob(f'{models_link}/*'):
        shutil.move(f, MODELS_DIR)
    shutil.rmtree(models_link)
if not os.path.islink(models_link):
    os.symlink(MODELS_DIR, models_link)

# ---- 3) Python env ----
READY_MARKER = f'{VENV_DIR}/.ready-{FACEFUSION_VERSION}'
if not os.path.isfile(READY_MARKER):
    if shutil.which('uv') is None:
        sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    shutil.rmtree(VENV_DIR, ignore_errors=True)
    sh(['uv', 'venv', '--python', '3.12', VENV_DIR])
    reqs = [l.strip() for l in open(f'{FF_DIR}/requirements.txt') if l.strip()
            and not l.startswith(('gradio', 'onnxruntime'))]
    # FaceFusion's installer maps "cuda@12" → onnxruntime-gpu 1.24.4; [cuda,cudnn] pulls CUDA 12 + cuDNN 9 wheels.
    sh(['uv', 'pip', 'install', '--python', FF_PY, *reqs, 'onnxruntime-gpu[cuda,cudnn]==1.24.4'])
    pathlib.Path(READY_MARKER).touch()
else:
    print('Python env already installed (skipping).')

sh([FF_PY, '-c', 'import onnxruntime, cv2, numpy; print("onnxruntime", onnxruntime.__version__, onnxruntime.get_available_providers())'], env=ff_env())
sh([FF_PY, 'facefusion.py', '--version'], env=ff_env(), cwd=FF_DIR)
print('✅ Dependencies ready')

In [ ]:
%%writefile /content/ff_tools.py
# ff_tools.py - small helpers that use FaceFusion's OWN code, so the notebook sees videos
# and faces exactly the way FaceFusion will during the real run.
#
#   python ff_tools.py <facefusion_dir> info     <video>
#   python ff_tools.py <facefusion_dir> prefetch <source> <target> [facefusion args...]
#   python ff_tools.py <facefusion_dir> cudacheck
#   python ff_tools.py <facefusion_dir> facemap  <source> <target> <frames,comma,separated> <out_dir> [facefusion args...]
#
# Every command prints exactly one JSON line prefixed with "@@JSON@@" for the notebook to parse.
import glob
import json
import os
import sys

FF_DIR, COMMAND = sys.argv[1], sys.argv[2]
REST = sys.argv[3:]
os.chdir(FF_DIR)
sys.path.insert(0, FF_DIR)


def emit(payload):
	print('@@JSON@@' + json.dumps(payload), flush = True)


def apply_facefusion_args(source, target, extra):
	from facefusion import state_manager
	from facefusion.args import apply_args
	from facefusion.program import create_program

	sys.argv = [ 'facefusion.py', 'headless-run', '-s', source, '-t', target, '-o', os.path.join(FF_DIR, '_unused.mp4') ] + extra
	args = vars(create_program().parse_args())
	apply_args(args, state_manager.init_item)


if COMMAND == 'info':
	from facefusion.vision import count_video_frame_total, detect_video_fps, detect_video_resolution
	video = REST[0]
	emit({ 'frame_total': count_video_frame_total(video), 'fps': detect_video_fps(video), 'resolution': list(detect_video_resolution(video)) })

elif COMMAND == 'prefetch':
	# Downloads (and hash-checks) exactly the models the chosen settings need - nothing more.
	from facefusion import state_manager
	from facefusion.processors.core import get_processors_modules
	apply_facefusion_args(REST[0], REST[1], REST[2:])
	ok = True
	for processor_module in get_processors_modules(state_manager.get_item('processors')):
		ok = processor_module.pre_check() and ok
	emit({ 'ok': bool(ok) })

elif COMMAND == 'cudacheck':
	# Proves onnxruntime can really run a model on the GPU (not just list the provider).
	import onnxruntime
	models = sorted(glob.glob(os.path.join(FF_DIR, '.assets', 'models', '*.onnx')), key = os.path.getsize)
	session = onnxruntime.InferenceSession(models[0], providers = [ 'CUDAExecutionProvider', 'CPUExecutionProvider' ])
	emit({ 'onnxruntime': onnxruntime.__version__, 'model': os.path.basename(models[0]), 'active_providers': session.get_providers() })

elif COMMAND == 'facemap':
	# Lists the faces in some frames in the SAME order FaceFusion uses for
	# --reference-face-position, and saves a PNG per frame with numbered boxes.
	import cv2
	from facefusion import face_classifier, face_detector, face_landmarker, face_recognizer
	from facefusion.face_creator import get_many_faces
	from facefusion.face_selector import sort_and_filter_faces
	from facefusion.vision import read_static_image, read_static_video_frame

	source, target, frames, out_dir = REST[0], REST[1], REST[2], REST[3]
	apply_facefusion_args(source, target, REST[4:])
	for module in (face_detector, face_landmarker, face_recognizer, face_classifier):
		if not module.pre_check():
			emit({ 'error': 'model download failed: ' + module.__name__ })
			sys.exit(1)

	os.makedirs(out_dir, exist_ok = True)
	source_faces = get_many_faces([ read_static_image(source) ])
	result = { 'source_faces': len(source_faces), 'frames': {} }

	for frame_number in [ int(f) for f in frames.split(',') if f != '' ]:
		frame = read_static_video_frame(target, frame_number)
		if frame is None:
			continue
		faces = sort_and_filter_faces(source_faces, get_many_faces([ frame ]))
		canvas = frame.copy()
		thickness = max(2, frame.shape[1] // 400)
		entries = []
		for position, face in enumerate(faces):
			x1, y1, x2, y2 = [ int(v) for v in face.bounding_box ]
			cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), thickness)
			cv2.putText(canvas, str(position), (x1, max(40, y1 - 12)), cv2.FONT_HERSHEY_SIMPLEX, thickness * 0.6, (0, 255, 0), thickness)
			entries.append({ 'position': position, 'box': [ x1, y1, x2, y2 ], 'area': (x2 - x1) * (y2 - y1), 'gender': face.gender, 'score': round(float(face.score_set.get('detector')), 3) })
		cv2.imwrite(os.path.join(out_dir, 'frame_%06d.png' % frame_number), canvas)
		result['frames'][str(frame_number)] = entries
	emit(result)

else:
	emit({ 'error': 'unknown command ' + COMMAND })
	sys.exit(2)

## 3 · Download / set up model weights

In [ ]:
#@title 3.1 · Download the models for the chosen settings + verify GPU inference { display-mode: "form" }
# Downloads (once, to persistent storage) and hash-checks only the ONNX models these settings need:
# face detector(s), landmarker(s), recognizer (identity embedding), classifier, occlusion/region masks,
# the face-swap model (and enhancer if enabled), and FaceFusion's content filter.
# Then it runs a real model on the GPU to prove CUDA works (rather than only listing the provider).
dummy_image = f'{LOCAL_WORK}/dummy.jpg'
if not os.path.isfile(dummy_image):
    sh(['ffmpeg', '-v', 'error', '-y', '-f', 'lavfi', '-i', 'color=c=gray:s=64x64', '-frames:v', '1', dummy_image])
result = ff_tool('prefetch', dummy_image, dummy_image, *facefusion_args(), quiet=False)
if not result.get('ok'):
    raise RuntimeError('❌ Model download failed. Check the log above, then re-run this cell (downloads resume).')
total = sum(os.path.getsize(f) for f in glob.glob(f'{MODELS_DIR}/*.onnx'))
print(f'✅ Models ready in {MODELS_DIR} ({total / 1e9:.2f} GB)')

check = ff_tool('cudacheck')
print('GPU inference check:', check)
if EXECUTION_PROVIDER == 'cuda' and 'CUDAExecutionProvider' not in check['active_providers']:
    raise RuntimeError('❌ onnxruntime could not run on the GPU. Restart the runtime and run all cells again.')
print('✅ CUDA execution confirmed' if EXECUTION_PROVIDER == 'cuda' else 'ℹ️ Running on CPU (slow).')

## 4 · 📤 UPLOAD #1 — target video (`target_video.mp4`)

Run the cell below and click **Choose Files** to pick your Diwali video.
* Already uploaded before? It's reused automatically. Tick `REPLACE_TARGET` to upload a new one.
* **Big file (> ~500 MB)?** Browser upload is slow. Copy it to Google Drive instead, at
  `MyDrive/diwali_faceswap/inputs/target_video.mp4`, and run the cell; it will find it there.

In [ ]:
#@title 📤 4.1 · UPLOAD TARGET VIDEO  ← upload target_video.mp4 here { display-mode: "form" }
REPLACE_TARGET = False  #@param {type:"boolean"}
# After upload the video is analysed, and a video-only working copy is prepared for FaceFusion:
#  • Constant-frame-rate, 8-bit, non-rotated videos are stream-copied (lossless, no re-encode).
#  • Variable frame rate (common on phones), rotation flags, HDR or 10-bit video are converted ONCE into a
#    near-lossless constant-FPS H.264 copy with the same displayed resolution and duration.
#  The ORIGINAL file stays untouched and its audio is used at the end.
existing = find_input('target_video')
if existing and REPLACE_TARGET:
    os.remove(existing); existing = None
if existing is None:
    from google.colab import files
    print('👉 Choose your target Diwali video...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded.')
    name = next(iter(uploaded))
    ext = os.path.splitext(name)[1].lower() or '.mp4'
    dest = f'{INPUT_DIR}/target_video{ext}'
    shutil.move(name, dest + '.part'); os.replace(dest + '.part', dest)
    existing = dest
TARGET_ORIGINAL = existing
print('Target video:', TARGET_ORIGINAL)

probe = ffprobe(TARGET_ORIGINAL)
v = next(s for s in probe['streams'] if s['codec_type'] == 'video')
a = next((s for s in probe['streams'] if s['codec_type'] == 'audio'), None)

def frac(x):
    try:
        f = Fraction(x)
        return f if f > 0 else None
    except (ZeroDivisionError, ValueError, TypeError):
        return None

r_fps, avg_fps = frac(v.get('r_frame_rate')), frac(v.get('avg_frame_rate'))
rotation = 0
for sd in v.get('side_data_list', []):
    if 'rotation' in sd:
        rotation = int(float(sd['rotation']))
rotation = rotation or int(v.get('tags', {}).get('rotate', 0))
is_vfr = bool(r_fps and avg_fps and abs(float(r_fps - avg_fps)) / float(r_fps) > 0.002)
is_hdr = v.get('color_transfer') in ('smpte2084', 'arib-std-b67')
pix_ok = v.get('pix_fmt') in ('yuv420p', 'yuvj420p')
w, h = int(v['width']), int(v['height'])
display_w, display_h = (h, w) if abs(rotation) % 180 == 90 else (w, h)

# Nominal FPS to keep. If r_fps and avg_fps are within 2% (phone jitter), keep r_fps (e.g. 30 or 30000/1001).
fps = r_fps if (r_fps and (not avg_fps or abs(float(r_fps - avg_fps)) / float(r_fps) < 0.02)) else (avg_fps or r_fps)
fps = fps.limit_denominator(1001)
STANDARD_FPS = (Fraction(24000, 1001), 24, 25, Fraction(30000, 1001), 30, 48, 50, Fraction(60000, 1001), 60, 120)
nearest = min(STANDARD_FPS, key=lambda std: abs(float(fps) - float(std)))
if abs(float(fps) - float(nearest)) / float(nearest) < 0.01:   # snap phone averages like 29.94 → 30
    fps = Fraction(nearest)
duration = float(probe['format']['duration'])
needs_convert = is_vfr or rotation != 0 or is_hdr or not pix_ok

sig = file_signature(TARGET_ORIGINAL)
TARGET_WORK = f'{LOCAL_WORK}/target_work.mp4'
meta = json.load(open(TARGET_META_JSON)) if os.path.isfile(TARGET_META_JSON) else {}
if meta.get('signature') != sig or not os.path.isfile(TARGET_WORK):
    if needs_convert:
        print(f'Converting once → constant {float(fps):.3f} fps, 8-bit H.264 (VFR={is_vfr}, rotation={rotation}, HDR={is_hdr}, pix_fmt={v.get("pix_fmt")})')
        base = ['ffmpeg', '-v', 'error', '-stats', '-y', '-i', TARGET_ORIGINAL, '-map', '0:v:0', '-an', '-sn', '-dn',
                '-fps_mode', 'cfr', '-r', str(fps), '-c:v', 'libx264', '-preset', 'medium', '-crf', '12']
        vf_tonemap = ('zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,tonemap=hable:desat=0,'
                      'zscale=t=bt709:m=bt709:r=tv,format=yuv420p')
        if is_hdr and sh(base + ['-vf', vf_tonemap, TARGET_WORK], check=False)[0] == 0:
            pass
        else:
            sh(base + ['-pix_fmt', 'yuv420p', TARGET_WORK])
    else:
        print('Video is constant-FPS 8-bit — stream-copying the video track (lossless).')
        sh(['ffmpeg', '-v', 'error', '-y', '-i', TARGET_ORIGINAL, '-map', '0:v:0', '-c:v', 'copy', '-an', '-sn', '-dn', TARGET_WORK])
    ff_info = ff_tool('info', TARGET_WORK)   # how FaceFusion itself sees the working copy
    meta = {
        'signature': sig, 'original': TARGET_ORIGINAL, 'converted': needs_convert,
        'fps_fraction': str(fps), 'fps': float(fps), 'duration': duration,
        'display_resolution': [display_w, display_h], 'codec': v.get('codec_name'),
        'has_audio': a is not None, 'audio_codec': a.get('codec_name') if a else None,
        'ff_frame_total': ff_info['frame_total'], 'ff_fps': ff_info['fps'], 'ff_resolution': ff_info['resolution'],
    }
    json.dump(meta, open(TARGET_META_JSON, 'w'), indent=2)
else:
    print('Working copy already prepared (skipping).')

print(json.dumps(meta, indent=2))
if meta['ff_resolution'] != meta['display_resolution']:
    print('⚠️ FaceFusion sees a different resolution than the displayed one:', meta['ff_resolution'])
if not meta['has_audio']:
    print('ℹ️ This video has no audio track; the output will be silent too.')
print('✅ Target video ready')

## 5 · 📤 UPLOAD #2 — source face photo (`source_face.jpg`)

Use a sharp, well-lit, **front-facing** photo of her, without sunglasses, hair off the face, face ideally ≥ 300 px across.
If a photo has several people in it, FaceFusion uses the **largest** face, so crop out anyone bigger than her.

**Optional (5.2):** add 1–3 more photos of her from **other angles** (¾ view, slight side view). FaceFusion averages her identity
across all photos, which usually makes the swap steadier when she turns her head.

In [ ]:
#@title 📤 5.1 · UPLOAD SOURCE FACE  ← upload source_face.jpg here { display-mode: "form" }
REPLACE_SOURCE = False  #@param {type:"boolean"}
existing = find_input('source_face')
if existing and REPLACE_SOURCE:
    os.remove(existing); existing = None
if existing is None:
    from google.colab import files
    print('👉 Choose the source face photo...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded.')
    name = next(iter(uploaded))
    ext = os.path.splitext(name)[1].lower()
    dest = f'{INPUT_DIR}/source_face.jpg'
    if ext in ('.jpg', '.jpeg'):
        shutil.move(name, dest)
    else:  # convert png / webp / heic-converted files etc. to high-quality jpg
        sh(['ffmpeg', '-v', 'error', '-y', '-i', name, '-q:v', '1', dest]); os.remove(name)
    existing = dest
SOURCE_FACE = existing
print('Source face:', SOURCE_FACE)

# Detect faces in the photo with FaceFusion's own detector.
check = ff_tool('facemap', SOURCE_FACE, TARGET_WORK, '', f'{LOCAL_WORK}/facemap', *facefusion_args())
n = check['source_faces']
if n == 0:
    raise SystemExit('❌ No face detected in the source photo. Use a clearer, front-facing photo (REPLACE_SOURCE = True).')
if n > 1:
    print(f'⚠️ {n} faces detected in the photo. FaceFusion uses the LARGEST one, so make sure that is her '
          '(otherwise crop the photo and re-upload with REPLACE_SOURCE ticked).')
else:
    print('✅ One face detected in the source photo')
display(Image(SOURCE_FACE, width=280))
SOURCE_FACES = [SOURCE_FACE] + sorted(glob.glob(f'{INPUT_DIR}/source_face_extra_*.jpg'))

In [ ]:
#@title 📤 5.2 · (Optional) ADD MORE PHOTOS OF HER — other angles help side views { display-mode: "form" }
ADD_EXTRA_PHOTOS = False    #@param {type:"boolean"}
CLEAR_EXTRA_PHOTOS = False  #@param {type:"boolean"}
# Tick ADD_EXTRA_PHOTOS and run this cell to upload 1-3 extra photos (you can select several at once).
# They are kept for later runs. Tick CLEAR_EXTRA_PHOTOS to remove them again.
if CLEAR_EXTRA_PHOTOS:
    for f in glob.glob(f'{INPUT_DIR}/source_face_extra_*.jpg'):
        os.remove(f)
    print('Removed extra photos.')
if ADD_EXTRA_PHOTOS:
    from google.colab import files
    print('👉 Choose one or more extra photos of her...')
    uploaded = files.upload()
    for name in uploaded:
        index = len(glob.glob(f'{INPUT_DIR}/source_face_extra_*.jpg')) + 1
        dest = f'{INPUT_DIR}/source_face_extra_{index}.jpg'
        if os.path.splitext(name)[1].lower() in ('.jpg', '.jpeg'):
            shutil.move(name, dest)
        else:
            sh(['ffmpeg', '-v', 'error', '-y', '-i', name, '-q:v', '1', dest]); os.remove(name)
        found = ff_tool('facemap', dest, TARGET_WORK, '', f'{LOCAL_WORK}/facemap', *facefusion_args())['source_faces']
        if found == 0:
            os.remove(dest)
            print(f'❌ {name}: no face detected, skipped.')
        else:
            print(f'✅ {name}: added' + (f' ({found} faces, the largest is used)' if found > 1 else ''))

SOURCE_FACES = [SOURCE_FACE] + sorted(glob.glob(f'{INPUT_DIR}/source_face_extra_*.jpg'))
print(f'Using {len(SOURCE_FACES)} photo(s) of her:')
for f in SOURCE_FACES:
    print('  ', os.path.basename(f))
if len(SOURCE_FACES) > 1:
    display(*[Image(f, width=160) for f in SOURCE_FACES])

## 6 · Preview / test configuration (5-second clip)

1. Picks a **reference frame** and shows every detected face with a **number**. The face numbered
   `REFERENCE_FACE_POSITION` (default 0, the largest) is the person who gets the new face, in every frame.
2. Renders a **5-second preview** (original | swapped, side by side) and estimates the full-run time.

Wrong person, or the face isn't swapped in some angles? Change the settings in **1.2**, re-run 1.2, then re-run this cell:
* Wrong person → set `REFERENCE_FACE_POSITION` to the right number (or set `REFERENCE_TIME_SECONDS` to a moment where she is clearly visible).
* Not swapped when the head turns → raise `REFERENCE_FACE_DISTANCE` (0.5–0.6) and/or lower `FACE_DETECTOR_SCORE` (0.35).
* Another person also gets swapped → lower `REFERENCE_FACE_DISTANCE` (0.3).
* **Only one person in the whole video** and some visible faces aren't swapped → set `FACE_SELECTOR_MODE = "one"`.
* Hands or objects over the face look wrong → keep `OCCLUSION_MASK` on. Hair over the forehead → enable `REGION_MASK`.
* **Side views / head turns look awkward** → run **6.2** to compare fixes side by side (region mask, expression restorer,
  a different swap model) and add angled photos in **5.2**. Profiles are the hardest case for every frame-based swapper:
  the models are trained mostly on frontal faces, so expect "better", not "perfect", on strong side views.

In [ ]:
#@title 6.1 · Choose the reference face + render the 5-second preview { display-mode: "form" }
meta = json.load(open(TARGET_META_JSON))
FPS, FRAMES = meta['ff_fps'], meta['ff_frame_total']
facemap_dir = f'{LOCAL_WORK}/facemap'
shutil.rmtree(facemap_dir, ignore_errors=True)

# ---- reference frame ----
if REFERENCE_TIME_SECONDS >= 0:
    candidates = [min(FRAMES - 1, int(round(REFERENCE_TIME_SECONDS * FPS)))]
else:  # auto: sample 16 frames, pick the one with the biggest face (most frontal/close shots)
    candidates = sorted({int(FRAMES * (i + 0.5) / 16) for i in range(16)})
fm = ff_tool('facemap', SOURCE_FACES[0], TARGET_WORK, ','.join(map(str, candidates)), facemap_dir, *facefusion_args())
scored = [(max(f['area'] for f in faces), int(n)) for n, faces in fm['frames'].items() if len(faces) > REFERENCE_FACE_POSITION]
if not scored:
    raise SystemExit(f'❌ No frame with at least {REFERENCE_FACE_POSITION + 1} face(s) found at the sampled times. '
                     'Set REFERENCE_TIME_SECONDS (in 1.2) to a moment where her face is clearly visible.')
REFERENCE_FRAME = max(scored)[1]
faces_at_ref = fm['frames'][str(REFERENCE_FRAME)]
json.dump({'frame': REFERENCE_FRAME, 'position': REFERENCE_FACE_POSITION}, open(REFERENCE_JSON, 'w'))

print(f'Reference frame: #{REFERENCE_FRAME} ({REFERENCE_FRAME / FPS:.2f}s). Faces found (0 = largest):')
for f in faces_at_ref:
    marker = '  ← will be swapped' if f['position'] == REFERENCE_FACE_POSITION else ''
    print(f"   [{f['position']}] box={f['box']} gender={f['gender']} score={f['score']}{marker}")
display(Image(f'{facemap_dir}/frame_{REFERENCE_FRAME:06d}.png', width=720))

# ---- 5-second preview ----
preview_frames = int(round(PREVIEW_SECONDS * FPS))
if PREVIEW_START_SECONDS >= 0:
    start = int(round(PREVIEW_START_SECONDS * FPS))
else:
    start = REFERENCE_FRAME - preview_frames // 2
start = max(0, min(start, FRAMES - preview_frames))
end = min(FRAMES, start + preview_frames)
preview_raw = f'{LOCAL_WORK}/preview_raw.mp4'
if os.path.exists(preview_raw):
    os.remove(preview_raw)
print(f'\nRendering preview: frames {start}–{end} ({start / FPS:.2f}s – {end / FPS:.2f}s)...')
json.dump({'frame': REFERENCE_FRAME, 'position': REFERENCE_FACE_POSITION, 'preview': [start, end]}, open(REFERENCE_JSON, 'w'))
elapsed = run_facefusion(SOURCE_FACES, TARGET_WORK, preview_raw, start, end, REFERENCE_FRAME)

# add the matching slice of the original audio, plus a side-by-side comparison
PREVIEW_OUT = f'{OUTPUT_DIR}/preview_{PREVIEW_SECONDS}s.mp4'
audio = ['-ss', f'{start / FPS:.6f}', '-t', f'{(end - start) / FPS:.6f}', '-i', meta['original']] if meta['has_audio'] else []
sh(['ffmpeg', '-v', 'error', '-y', '-i', preview_raw, *audio, '-map', '0:v:0', *(['-map', '1:a:0', '-c:a', 'aac', '-b:a', '192k'] if audio else []),
    '-c:v', 'copy', '-movflags', '+faststart', PREVIEW_OUT])
compare = f'{LOCAL_WORK}/preview_compare.mp4'
sh(['ffmpeg', '-v', 'error', '-y', '-ss', f'{start / FPS:.6f}', '-t', f'{(end - start) / FPS:.6f}', '-i', TARGET_WORK, '-i', PREVIEW_OUT,
    '-filter_complex', '[0:v]scale=-2:480,setsar=1[a];[1:v]scale=-2:480,setsar=1[b];[a][b]hstack=inputs=2[v]',
    '-map', '[v]', *(['-map', '1:a:0'] if audio else []), '-c:v', 'libx264', '-crf', '23', '-preset', 'veryfast', '-c:a', 'aac', compare])

per_frame = elapsed / max(1, end - start)
print(f'\n⏱ Preview: {end - start} frames in {elapsed:.0f}s ({1 / per_frame:.1f} fps incl. model loading).')
print(f'   Estimated full video: ~{per_frame * FRAMES / 60:.0f} min for {FRAMES} frames.')
print('   LEFT = original, RIGHT = swapped. Saved:', PREVIEW_OUT)
show_video(compare)

if not CONTINUE_AFTER_PREVIEW:
    raise SystemExit('⏸ Stopped after preview (CONTINUE_AFTER_PREVIEW = False). Adjust settings or set it True and run section 7.')

In [ ]:
#@title 6.2 · (Optional) Compare fixes for side views — same clip, 4 settings side by side { display-mode: "form" }
RUN_COMPARISON = False        #@param {type:"boolean"}
COMPARE_START_SECONDS = -1.0  #@param {type:"number"}
COMPARE_SECONDS = 3           #@param {type:"integer"}
# Tick RUN_COMPARISON and run this cell. Set COMPARE_START_SECONDS to just before she turns her head
# (-1 = same window as the preview). Takes about 4x the preview time. Grid layout:
#   original | A | B
#      C     | D | (blank)
# Pick the best one, copy its settings into cell 1.2, re-run 1.2, then continue with section 7.
PRESETS = {
    'A · current settings': {},
    'B · region mask + softer edges': {'REGION_MASK': True, 'FACE_MASK_BLUR': 0.5},
    'C · B + expression restorer': {'REGION_MASK': True, 'FACE_MASK_BLUR': 0.5, 'EXPRESSION_RESTORER': True},
    'D · C with inswapper model': {'REGION_MASK': True, 'FACE_MASK_BLUR': 0.5, 'EXPRESSION_RESTORER': True,
                                   'FACE_SWAPPER_MODEL': 'inswapper_128_fp16'},
}

if not RUN_COMPARISON:
    print('Skipped (tick RUN_COMPARISON to compare settings).')
else:
    meta = json.load(open(TARGET_META_JSON))
    FPS, FRAMES = meta['ff_fps'], meta['ff_frame_total']
    ref = json.load(open(REFERENCE_JSON))
    n_frames = int(round(COMPARE_SECONDS * FPS))
    c_start = int(round(COMPARE_START_SECONDS * FPS)) if COMPARE_START_SECONDS >= 0 else ref['preview'][0]
    c_start = max(0, min(c_start, FRAMES - n_frames))
    c_end = min(FRAMES, c_start + n_frames)
    print(f'Comparing frames {c_start}–{c_end} ({c_start / FPS:.2f}s – {c_end / FPS:.2f}s)')

    clips = []
    for i, (label, overrides) in enumerate(PRESETS.items()):
        saved = {k: globals()[k] for k in overrides}
        globals().update(overrides)
        try:
            out = f'{LOCAL_WORK}/compare_{i}.mp4'
            if os.path.exists(out):
                os.remove(out)
            print(f'\n▶ {label}')
            took = run_facefusion(SOURCE_FACES, TARGET_WORK, out, c_start, c_end, ref['frame'])
            print(f'   {took:.0f}s')
            clips.append(out)
        finally:
            globals().update(saved)

    grid = f'{OUTPUT_DIR}/compare_settings.mp4'
    t = ['-ss', f'{c_start / FPS:.6f}', '-t', f'{(c_end - c_start) / FPS:.6f}']
    inputs = [*t, '-i', TARGET_WORK]
    for c in clips:
        inputs += ['-i', c]
    names = ['original'] + list(PRESETS)

    def build(with_labels):
        parts = []
        for k in range(5):
            label = ''
            if with_labels:
                text = names[k].replace(':', ' ').replace("'", '')
                label = f",drawtext=text='{text}':x=10:y=10:fontsize=22:fontcolor=white:box=1:boxcolor=black@0.6"
            parts.append(f'[{k}:v]scale=-2:360,setsar=1,format=yuv420p{label}[v{k}]')
        parts.append('[v0]split[v0a][v0b];[v0b]drawbox=c=black:t=fill[blank]')
        parts.append('[v0a][v1][v2]hstack=3[top];[v3][v4][blank]hstack=3[bottom];[top][bottom]vstack[v]')
        return ';'.join(parts)

    cmd = ['ffmpeg', '-v', 'error', '-y', *inputs, '-map', '[v]', '-c:v', 'libx264', '-crf', '23', '-preset', 'veryfast']
    if sh(cmd + ['-filter_complex', build(True), grid], check=False, quiet=True)[0] != 0:
        sh(cmd + ['-filter_complex', build(False), grid])   # ffmpeg without drawtext: no labels
    print('\nLayout:  original | A | B   /   C | D | (blank)')
    for name, overrides in PRESETS.items():
        print(f'  {name}: {overrides or "as configured in 1.2"}')
    show_video(grid, width=1000)

## 7 · Run the face swap on the full video

The video is processed in chunks of `CHUNK_SECONDS`. Each finished chunk is saved to the persistent folder.
**If Colab disconnects, run all cells again** and finished chunks are skipped. If you change any setting, old chunks are discarded
automatically so the output stays consistent.

In [ ]:
#@title 7.1 · Face-swap all frames (resumable) { display-mode: "form" }
meta = json.load(open(TARGET_META_JSON))
FPS, FRAMES = meta['ff_fps'], meta['ff_frame_total']
REFERENCE_FRAME = json.load(open(REFERENCE_JSON))['frame']

# Fingerprint of everything that affects the pixels. If it changes, old chunks are stale.
fingerprint = hashlib.sha256(json.dumps([FACEFUSION_VERSION, meta['signature'], [file_signature(f) for f in SOURCE_FACES],
                                         pixel_affecting_args(facefusion_args(REFERENCE_FRAME)), CHUNK_SECONDS]).encode()).hexdigest()[:16]
manifest_path = f'{CHUNKS_DIR}/manifest.json'
manifest = json.load(open(manifest_path)) if os.path.isfile(manifest_path) else {}
if manifest.get('fingerprint') != fingerprint:
    if glob.glob(f'{CHUNKS_DIR}/chunk_*.mp4'):
        print('Settings or inputs changed → discarding old chunks.')
    for f in glob.glob(f'{CHUNKS_DIR}/chunk_*.mp4'):
        os.remove(f)
    manifest = {'fingerprint': fingerprint, 'done': {}}
    json.dump(manifest, open(manifest_path, 'w'))

chunk_frames = max(1, int(round(CHUNK_SECONDS * FPS)))
ranges = [(s, min(FRAMES, s + chunk_frames)) for s in range(0, FRAMES, chunk_frames)]
print(f'{FRAMES} frames @ {FPS:.3f} fps → {len(ranges)} chunk(s) of up to {chunk_frames} frames')

t0 = time.time()
for i, (s, e) in enumerate(ranges):
    final_chunk = f'{CHUNKS_DIR}/chunk_{i:04d}.mp4'
    if os.path.isfile(final_chunk) and manifest['done'].get(str(i)) == [s, e]:
        print(f'[{i + 1}/{len(ranges)}] frames {s}–{e}: already done ✓')
        continue
    print(f'[{i + 1}/{len(ranges)}] frames {s}–{e} ({s / FPS:.1f}s–{e / FPS:.1f}s) ...')
    local_chunk = f'{LOCAL_WORK}/chunk_tmp.mp4'
    if os.path.exists(local_chunk):
        os.remove(local_chunk)
    took = run_facefusion(SOURCE_FACES, TARGET_WORK, local_chunk, s, e, REFERENCE_FRAME)
    got = count_frames(local_chunk)
    if got != e - s:
        print(f'   ⚠️ chunk has {got} frames, expected {e - s}')
    shutil.copy(local_chunk, final_chunk + '.part')
    os.replace(final_chunk + '.part', final_chunk)         # atomic: half-written chunks never count as done
    manifest['done'][str(i)] = [s, e]
    json.dump(manifest, open(manifest_path, 'w'))
    done = sum(b - a for a, b in ranges[:i + 1])
    print(f'   ✓ {took:.0f}s ({(e - s) / took:.1f} fps) — overall {100 * done / FRAMES:.0f}%')

print(f'✅ All chunks done ({(time.time() - t0) / 60:.1f} min this session)')

## 8 · Recombine the original audio

In [ ]:
#@title 8.1 · Join chunks + mux original audio → /content/swapped_video.mp4 { display-mode: "form" }
# Chunks are joined without re-encoding (stream copy). The audio comes from the ORIGINAL upload:
# copied bit-for-bit if it's AAC/MP3, otherwise encoded to AAC 320k (MP4-compatible).
meta = json.load(open(TARGET_META_JSON))
manifest = json.load(open(f'{CHUNKS_DIR}/manifest.json'))
chunks = [f'{CHUNKS_DIR}/chunk_{int(i):04d}.mp4' for i in sorted(manifest['done'], key=int)]
list_file = f'{LOCAL_WORK}/chunks.txt'
with open(list_file, 'w') as fh:
    for c in chunks:
        fh.write(f"file '{c}'\n")

audio_args = []
if meta['has_audio']:
    acodec = ['-c:a', 'copy'] if meta['audio_codec'] in ('aac', 'mp3') else ['-c:a', 'aac', '-b:a', '320k']
    audio_args = ['-i', meta['original'], '-map', '1:a:0', *acodec]
sh(['ffmpeg', '-v', 'error', '-y', '-f', 'concat', '-safe', '0', '-i', list_file, *audio_args,
    '-map', '0:v:0', '-c:v', 'copy', '-movflags', '+faststart', FINAL_OUTPUT])

# ---- verify that duration / fps / resolution / audio are preserved ----
out = ffprobe(FINAL_OUTPUT)
ov = next(s for s in out['streams'] if s['codec_type'] == 'video')
oa = next((s for s in out['streams'] if s['codec_type'] == 'audio'), None)
out_frames = count_frames(FINAL_OUTPUT)
rows = [
    ('Resolution', f"{meta['display_resolution'][0]}x{meta['display_resolution'][1]}", f"{ov['width']}x{ov['height']}"),
    ('FPS', f"{meta['fps']:.3f}", f"{float(Fraction(ov['r_frame_rate'])):.3f}"),
    ('Frames', str(meta['ff_frame_total']), str(out_frames)),
    ('Duration (s)', f"{meta['duration']:.2f}", f"{float(out['format']['duration']):.2f}"),
    ('Audio', meta['audio_codec'] or 'none', oa['codec_name'] if oa else 'none'),
]
print(f"{'':14}{'original':>14}{'swapped':>14}")
for name, a_val, b_val in rows:
    print(f'{name:14}{a_val:>14}{b_val:>14}')
if out_frames != meta['ff_frame_total']:
    print('⚠️ Frame count differs slightly. Check the result before sharing.')

shutil.copy(FINAL_OUTPUT, f'{OUTPUT_DIR}/swapped_video.mp4')
print(f'\n✅ Saved {FINAL_OUTPUT} ({os.path.getsize(FINAL_OUTPUT) / 1e6:.1f} MB)  + copy in {OUTPUT_DIR}')

## 9 · Download the final video

In [ ]:
#@title 9.1 · Download swapped_video.mp4 { display-mode: "form" }
# Starts a browser download right away; the button lets you download again later.
# For big files the Drive copy (MyDrive/diwali_faceswap/outputs/swapped_video.mp4) is often faster.
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

def _download(_=None):
    files.download(FINAL_OUTPUT)

button = widgets.Button(description='⬇️ Download swapped_video.mp4', button_style='success', layout=widgets.Layout(width='320px'))
button.on_click(_download)
display(button)
print(f'File: {FINAL_OUTPUT} ({os.path.getsize(FINAL_OUTPUT) / 1e6:.1f} MB)')
_download()

### Optional: watch a 20-second clip of the result here

In [ ]:
#@title 9.2 · Inline playback (first 20 s, downscaled) { display-mode: "form" }
clip = f'{LOCAL_WORK}/result_clip.mp4'
sh(['ffmpeg', '-v', 'error', '-y', '-i', FINAL_OUTPUT, '-t', '20', '-vf', 'scale=-2:540', '-c:v', 'libx264', '-crf', '24',
    '-preset', 'veryfast', '-c:a', 'aac', clip])
show_video(clip)